### Census Data Extraction

In [1]:
import numpy as np
import pandas as pd

#Change city
city = 'Texas'

#Census tables
table_dic = {
    'B09001':list('_0'+i+'E' for i in ['01']),
    'B11012':list('_0'+i+'E' for i in ['10','15']),
    'B16005':list('_0'+i+'E' for i in ['01','07','08','12','13','17','18','22','23','29','30','34','35','39','40','44','45']),
    'B26001':list('_0'+i+'E' for i in ['01']),
    'DP02':list('_00'+i+'E' for i in ['01','72P']),
    'DP03':list('_00'+i+'E' for i in ['09P']),
    'DP04':list('_00'+i+'E' for i in ['01','02','12','13','78','79','14P','58P']),
    'DP05':list('_00'+i+'E' for i in ['71','78','79','80','81','82','83']),
    'S0101':list('_C02_0'+i+'E' for i in ['30']),
    'S0601':list('_C01_0'+i+'E' for i in ['01','33']),
    'S1701':list('_C01_0'+i+'E' for i in ['01','40']),
    'S2503':list('_C01_0'+i+'E' for i in ['01','28','32','36','40']),
    'S2701':list('_C05_0'+i+'E' for i in ['01'])}

#Data extraction
census_df = pd.read_csv(city + '_S0601.csv', low_memory=False)[['NAME']]
census_df = census_df.drop(index=[0,1])

for key,value in table_dic.items():
    df = pd.read_csv(city + '_' + key + '.csv', low_memory=False)[['NAME'] + list(key+i for i in value)]
    df = df.drop(index=[0,1])
    df = df.replace('-', np.nan)
    df.iloc[:,1:] = df.iloc[:,1:].applymap(lambda x: float(x))
    if census_df['NAME'].equals(df['NAME']):
        for i in value:
            census_df[key+i] = df[key+i]

#Data output
census_df.to_csv(city + '_census.csv',index=False)

### SVI Dataset

In [2]:
import numpy as np
import pandas as pd

#Change city
city = 'Texas'

df = pd.read_csv(city + '_census.csv')
SVI = df[['NAME']].copy()

#Social vulnerability features calculation
SV_dic = {
    'EPL_POV150': df['S1701_C01_040E'] / df['S1701_C01_001E'] * 100,
    'EPL_UNEMP': df['DP03_0009PE'],
    'EPL_HBURD': df[['S2503_C01_0'+i+'E' for i in ['28','32','36','40']]].sum(axis=1) / df['S2503_C01_001E'] * 100,
    'EPL_NOHSDP': df['S0601_C01_033E'],
    'EPL_UNINSUR': df['S2701_C05_001E'],
    'EPL_AGE65': df['S0101_C02_030E'],
    'EPL_AGE17': df['B09001_001E'] / df['S0601_C01_001E'] * 100,
    'EPL_DISABL': df['DP02_0072PE'],
    'EPL_SNGPNT': df[['B11012_010E','B11012_015E']].sum(axis=1) / df['DP02_0001E'] * 100,
    'EPL_LIMENG': df[['B16005_0'+i+'E' for i in ['07','08','12','13','17','18','22','23','29','30','34','35','39','40','44','45']]].sum(axis=1) / df['B16005_001E'] * 100,
    'EPL_MINRTY': df[['DP05_00'+i+'E' for i in ['71','78','79','80','81','82','83']]].sum(axis=1) / df['S0601_C01_001E'] * 100,
    'EPL_MUNIT': df[['DP04_0012E','DP04_0013E']].sum(axis=1) / df['DP04_0001E'] * 100,
    'EPL_MOBILE': df['DP04_0014PE'],
    'EPL_CROWD': df[['DP04_0078E','DP04_0079E']].sum(axis=1) / df['DP04_0002E'] * 100,
    'EPL_NOVEH': df['DP04_0058PE'],
    'EPL_GROUPQ': df['B26001_001E'] / df['S0601_C01_001E'] * 100}

def percent_rank(pd_series):
    return list(np.round((pd_series < value).astype(int).sum()/(len(pd_series) -1), 4) for value in pd_series)

for key,value in SV_dic.items(): 
    SVI[key] = percent_rank(value)

SVI['RPL_THEME1'] = percent_rank(SVI.iloc[:,1:6].sum(axis=1))
SVI['RPL_THEME2'] = percent_rank(SVI.iloc[:,6:11].sum(axis=1))
SVI['RPL_THEME4'] = percent_rank(SVI.iloc[:,12:17].sum(axis=1))

#Drop rows
SVI = SVI.replace(np.inf, np.nan)
SVI = SVI.dropna()

#Data output
SVI.to_csv('SVI_Dataset.csv',index=False)